# 01 — Data Exploration  (Phase 1)

Visual sanity checks for the entire Phase 1 data pipeline.

**What this covers:**
1. Dataset stats — scene count, sample distribution by city
2. Camera intrinsics — K matrix visualised
3. Front-camera image load
4. Calibration chain — sensor → ego → world transforms
5. Map BEV GT mask — drivable area rasterised from HD map
6. Box BEV GT mask — vehicles and pedestrians from 3D boxes
7. Combined BEV GT — all channels overlaid
8. Dataset item — full `__getitem__` output inspected
9. DataLoader batch — shapes and timing
10. Boston vs Singapore GT comparison — terrain difference visible


In [ ]:
import sys
from pathlib import Path

# Make src/ importable from notebook
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
import torch

print('Imports OK')

## 1. Dataset Statistics

In [ ]:
from data.nuscenes_loader import (
    get_nusc, get_all_scene_tokens, get_all_sample_tokens,
    iterate_scene_samples, get_scene_location,
)

nusc         = get_nusc(verbose=True)
scene_tokens = get_all_scene_tokens()
all_tokens   = get_all_sample_tokens()

print(f'\nScenes : {len(scene_tokens)}')
print(f'Samples: {len(all_tokens)}')

# Samples per city
from collections import Counter
city_counter = Counter()
for sc in scene_tokens:
    loc = get_scene_location(sc)
    n   = sum(1 for _ in iterate_scene_samples(sc))
    city_counter[loc] += n

print('\nSamples per city:')
for city, n in city_counter.items():
    print(f'  {city}: {n}')

In [ ]:
# Bar chart: samples per city
fig, ax = plt.subplots(figsize=(7, 3))
cities  = list(city_counter.keys())
counts  = list(city_counter.values())
ax.bar(cities, counts, color=['steelblue','coral'])
ax.set_ylabel('Number of samples')
ax.set_title('nuScenes mini — samples per city')
plt.tight_layout()
plt.show()

## 2. Camera Intrinsics (K matrix)

In [ ]:
from data.nuscenes_loader import get_camera_data

token    = all_tokens[0]
cam_data = get_camera_data(token)
K        = cam_data['K']

print('K (camera intrinsic matrix):')
print(K)
print(f'\nfx = {K[0,0]:.2f}  fy = {K[1,1]:.2f}')
print(f'cx = {K[0,2]:.2f}  cy = {K[1,2]:.2f}')
print(f'\nT_cam2ego:\n{cam_data["T_cam2ego"]}')
print(f'\nT_ego2world:\n{cam_data["T_ego2world"]}')

## 3. Front-Camera Image

In [ ]:
bgr   = cv2.imread(str(cam_data['image_path']))
rgb   = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

print(f'Raw image shape: {rgb.shape}   dtype: {rgb.dtype}')

plt.figure(figsize=(12, 5))
plt.imshow(rgb)
plt.title(f'Front Camera — token: {token[:8]}...')
plt.axis('off')
plt.show()

## 4. Calibration Chain Sanity Check

In [ ]:
from data.calibration import (
    validate_intrinsics, validate_transform,
    build_cam_to_world, build_world_to_cam,
)

validate_intrinsics(cam_data['K'])
validate_transform(cam_data['T_cam2ego'],   'T_cam2ego')
validate_transform(cam_data['T_ego2world'], 'T_ego2world')

T_c2w = build_cam_to_world(cam_data['T_cam2ego'], cam_data['T_ego2world'])
T_w2c = build_world_to_cam(cam_data['T_cam2ego'], cam_data['T_ego2world'])

product = T_c2w @ T_w2c
max_err = np.abs(product - np.eye(4)).max()
print(f'K validation       : PASSED')
print(f'T_cam2ego validation: PASSED')
print(f'T_ego2world valid  : PASSED')
print(f'T_cam2world @ T_world2cam max error: {max_err:.2e}  (should be ~0)')

## 5. Map BEV GT Mask (Static Structure)

In [ ]:
from data.bev_gt_generator import generate_map_bev_mask
from utils.config import BEV, CLASSES

map_mask = generate_map_bev_mask(token)
print(f'Map mask shape : {map_mask.shape}  dtype: {map_mask.dtype}')
print(f'Unique values  : {np.unique(map_mask).tolist()}')
print(f'Drivable pixels: {(map_mask == 0).sum()}')

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(map_mask, cmap='tab10', vmin=0, vmax=9, origin='upper')
ax.set_title('Map BEV GT — drivable area (green=class 0)')
ax.set_xlabel('BEV x (lateral →)')
ax.set_ylabel('BEV y (forward ↑)')
# Mark ego position
cx = BEV['size'] // 2
ax.plot(cx, cx, 'y*', markersize=12, label='Ego')
ax.legend()
plt.show()

## 6. Box BEV GT Masks (Dynamic Objects)

In [ ]:
from data.bev_gt_generator import generate_box_bev_masks
from data.class_mappings import VEHICLE, PEDESTRIAN

box_masks = generate_box_bev_masks(token)
print(f'Box masks shape: {box_masks.shape}')
print(f'Vehicle pixels : {box_masks[VEHICLE].sum()}')
print(f'Pedestrian pixels: {box_masks[PEDESTRIAN].sum()}')

class_names = CLASSES['names']
fig, axes = plt.subplots(1, CLASSES['num_classes'], figsize=(14, 4))
for i, ax in enumerate(axes):
    ax.imshow(box_masks[i], cmap='hot', vmin=0, vmax=1, origin='upper')
    ax.set_title(f'Channel {i}: {class_names[i]}')
    cx = BEV['size'] // 2
    ax.plot(cx, cx, 'b*', markersize=8)
plt.suptitle('Box BEV GT — one channel per class')
plt.tight_layout()
plt.show()

## 7. Combined BEV GT (All Channels Overlaid)

In [ ]:
from data.bev_gt_generator import generate_bev_gt
from utils.config import CLASSES

bev_gt = generate_bev_gt(token)   # (C, H, W) float32
print(f'BEV GT shape : {bev_gt.shape}  dtype: {bev_gt.dtype}')
print(f'Value range  : [{bev_gt.min():.1f}, {bev_gt.max():.1f}]')
for i, name in enumerate(CLASSES['names']):
    print(f'  {name}: {int(bev_gt[i].sum())} positive pixels')

# RGB overlay: road=green, vehicle=red, pedestrian=blue
colors = np.array([
    [100, 160, 100],   # drivable — green
    [200,  80,  50],   # vehicle  — red/orange
    [ 50, 120, 220],   # pedestrian — blue
], dtype=np.float32)

H, W   = bev_gt.shape[1], bev_gt.shape[2]
canvas = np.zeros((H, W, 3), dtype=np.float32)
for c in range(bev_gt.shape[0]):
    mask = bev_gt[c] > 0.5
    canvas[mask] = colors[c] / 255.0

# Ego marker
cx = BEV['size'] // 2

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(rgb)
axes[0].set_title('Front Camera')
axes[0].axis('off')

axes[1].imshow(canvas, origin='upper')
axes[1].plot(cx, cx, 'y*', markersize=12)
axes[1].set_title('BEV Ground Truth')
axes[1].set_xlabel('← left | right →')
axes[1].set_ylabel('← behind | forward →')

patches = [
    mpatches.Patch(color=colors[i]/255, label=CLASSES['names'][i])
    for i in range(len(CLASSES['names']))
]
patches.append(mpatches.Patch(color='yellow', label='Ego vehicle'))
axes[1].legend(handles=patches, loc='lower right', fontsize=8)

plt.suptitle(f'Sample: {token[:8]}...')
plt.tight_layout()
plt.show()

## 8. Dataset __getitem__ Inspection

In [ ]:
from data.nuscenes_dataset import NuScenesBEVDataset

ds   = NuScenesBEVDataset(split='train')
item = ds[0]

print('Dataset item keys and shapes:')
for k, v in item.items():
    if hasattr(v, 'shape'):
        print(f'  {k:20s}: shape={tuple(v.shape)}  dtype={v.dtype}')
    else:
        print(f'  {k:20s}: {v}')

print(f'\nImage range  : [{item["image"].min():.3f}, {item["image"].max():.3f}]')
print(f'BEV GT range : [{item["bev_gt"].min():.1f}, {item["bev_gt"].max():.1f}]')

## 9. DataLoader Batch — Shapes and Timing

In [ ]:
import time
from data.nuscenes_dataset import build_dataloader

loader = build_dataloader(split='val', batch_size=2, shuffle=False)

t0    = time.time()
batch = next(iter(loader))
t1    = time.time()

print(f'Batch loaded in {(t1-t0)*1000:.0f} ms')
print('Batch tensor shapes:')
for k, v in batch.items():
    if hasattr(v, 'shape'):
        print(f'  {k:20s}: {tuple(v.shape)}')
    else:
        print(f'  {k:20s}: {v}')

## 10. Boston vs Singapore GT Comparison

In [ ]:
from data.nuscenes_loader import (
    get_all_scene_tokens, iterate_scene_samples, get_scene_location
)
from data.bev_gt_generator import generate_bev_gt
import cv2

# Find one Boston and one Singapore sample
boston_tok = singapore_tok = None
for sc in get_all_scene_tokens():
    loc = get_scene_location(sc)
    tok = next(iterate_scene_samples(sc))
    if 'boston' in loc and boston_tok is None:
        boston_tok = tok
    if 'singapore' in loc and singapore_tok is None:
        singapore_tok = tok
    if boston_tok and singapore_tok:
        break

def make_bev_rgb(token):
    gt = generate_bev_gt(token)   # (C, H, W)
    canvas = np.zeros((gt.shape[1], gt.shape[2], 3), dtype=np.float32)
    _colors = [[100,160,100],[200,80,50],[50,120,220]]
    for c in range(gt.shape[0]):
        canvas[gt[c]>0.5] = np.array(_colors[c]) / 255.0
    return canvas

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(make_bev_rgb(boston_tok), origin='upper')
axes[0].set_title('Boston Seaport BEV GT')
axes[0].plot(100, 100, 'y*', markersize=10)

axes[1].imshow(make_bev_rgb(singapore_tok), origin='upper')
axes[1].set_title('Singapore BEV GT')
axes[1].plot(100, 100, 'y*', markersize=10)

plt.suptitle('City comparison — same pipeline, different terrain')
plt.tight_layout()
plt.show()

print('\nNote: Boston has ramps + speed bumps → IPM will distort there.')
print('Singapore is flatter → IPM more accurate. This is our eval split.')

## Summary

If all cells ran without error and the BEV GT plots show:
- Green drivable area centred ahead of ego ✓
- Orange/red vehicle footprints near the road ✓
- Blue pedestrian footprints where pedestrians exist ✓

→ **Phase 1 pipeline is correct. Run `python scripts/verify_phase1.py` and proceed to Phase 2.**
